# Lab 12: Group Summit Gear's Customers

You are the analyst at **Summit Gear Co.** For two weeks you predicted something about these
2,200 accounts: how long each takes to pay, then which ones default. This week there is no
target. The controller asks two questions. Which accounts look alike? And which fit no group
at all? Groups help the credit and sales teams set terms and plan reviews. An account that fits
nowhere is worth a look before anyone signs off on the receivables. An AI tool can write the
clustering code, or you can type it from the Lab 12 page. You check that each fit used the columns, scaling and settings you asked for,
say what the clusters mean, and decide which unusual accounts deserve a question.

> **First:** in Colab, choose **File > Save a copy in Drive** and work in your copy. The
> copy that opens from the link does not save your work.

> **Then:** **Runtime > Run all.** The notebook runs clean before you have written
> anything. The **Final check** at the bottom lists what is still yours to do.

Plan on about 2.5 hours. The [Lab 12 page](https://seanmccaman.com/acctg6155/2026-fall/week-12/lab/lab_12.html) has the AI tutor, the data dictionary,
what each written cell is graded on, and what to submit.

## How this lab works

1. **You specify.** The columns, whether to scale, the number of clusters k, and the seed.
2. **You or the AI write the code.** Ask Gemini in Colab or any AI tool, or copy the code from the [Lab 12 page](https://seanmccaman.com/acctg6155/2026-fall/week-12/lab/lab_12.html), and paste it into the cell marked TODO.
3. **You check it ran.** `check_fit()` prints the rows used, the columns, whether the data was scaled, k, `n_init`, the seed and the cluster sizes. It warns if the model was not fit on the X you pass it. Compare each line with what you asked for.
4. **You interpret.** What the clusters mean for the credit team, in your own words.

Keep the names the TODO cells give you (`pair_raw`, `spend_share`, `X_raw`, `X_scaled`,
`pair_scaled`, `km_unscaled`, `km_scaled`, `elbow`, `profile`, `default_by_cluster`,
`dist_to_center`, `flags`, `flag_default_rate`). The Final check reads them by name.

## Section 0: Setup and load

Colab already has `pandas` and `scikit-learn`. On your own machine, run
`pip install pandas scikit-learn matplotlib` once first.

In [ ]:
# Section 0: run once.
import pandas as pd
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("pandas", pd.__version__, "  scikit-learn", sklearn.__version__)
print("setup ok")

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/week-11/summit_gear_customers.csv"

df = pd.read_csv(DATA_URL)
print("accounts:", df.shape)
print("missing values:", int(df.isna().sum().sum()), "  duplicate customer_id:", int(df["customer_id"].duplicated().sum()))

**No target this week.** The same table as Weeks 10 and 11, with its two outcome columns,
`defaulted` and `days_to_pay`, set aside. k-means never sees them. `defaulted` comes back once,
after the clusters are built, as a check on what they found. The
[data dictionary](https://github.com/sean-mccaman/acctg5150-090/blob/main/2026-fall/week-11/summit_gear_customers_dictionary.md) has every column, its unit and its timing. It was written for Week 11.
This week no column is a target, and `defaulted` comes back only for the check after the fact.

In [ ]:
# Set the two outcome columns aside. They come back in Exercise 2's check after the fact.
held_back = df[["defaulted", "days_to_pay"]].copy()
df = df.drop(columns=["defaulted", "days_to_pay"])
print("columns k-means could see:", list(df.columns))

**The six lab columns.** `annual_spend`, `prior_late_count`, `days_since_last_order`, `orders_per_year`, `discount_rate`, `avg_order_value`. They are the deck's eight behavior columns without
`credit_limit` and `avg_balance`. Exercise 2 asks what that choice changes. The settings are fixed
so everyone's clusters compare: k-means with `n_init=10` and `random_state=12`, and k = 4 for the
graded fit.

In [ ]:
LAB_COLUMNS = ['annual_spend', 'prior_late_count', 'days_since_last_order', 'orders_per_year', 'discount_rate', 'avg_order_value']
K = 4          # clusters in the graded fit
N_INIT = 10    # k-means restarts; it keeps the best of the ten
SEED = 12       # random_state, so every run gives the same clusters

print(df[LAB_COLUMNS].describe().round(2).T)

**Two provided helpers.** Run the cell below once. You do not edit it.

- `cluster_numbers(model)` numbers a fit's clusters 1 to k, with 1 the cluster that has the highest median `annual_spend`. k-means numbers clusters in whatever order it finds them, so this keeps your numbers and the key's in line. A cluster number is a label. It is not a ranking.
- `check_fit(model, X)` checks a fit against the X you pass it: rows, columns, scaled or not, k, `n_init`, the seed and the sizes. Pass the same X the model was fit on. If they don't match, it prints a warning instead. Run it after every fit.

In [ ]:
# @title Provided helpers: cluster_numbers() and check_fit() (run it; you do not edit it)

def renumber(labels, spend):
    """Cluster numbers 1 to k, 1 = the cluster with the highest median annual_spend.

    Takes:
      labels -- the model's cluster labels, one per account (0 to k-1, in the model's own order).
      spend  -- annual_spend, one value per account, in the same order.
    Returns:
      numpy array -- the cluster number for each account, 1 to k.
    """
    import numpy as np
    import pandas as pd
    labels = np.asarray(labels)
    med = pd.Series(np.asarray(spend, dtype=float)).groupby(labels).median().sort_values(ascending=False)
    remap = {old: i + 1 for i, old in enumerate(med.index)}
    return np.array([remap[v] for v in labels])


def cluster_numbers(model):
    """The cluster number of every account, 1 to k, 1 = the cluster with the highest median annual_spend."""
    return renumber(model.labels_, df["annual_spend"])


def check_fit(model, X):
    """Check a k-means fit against the X you pass it. Pass the same X the model was fit on."""
    Xa = np.asarray(X, dtype=float)
    cols = list(X.columns) if hasattr(X, "columns") else None
    p = model.get_params()
    print("rows used:   ", Xa.shape[0], "(should be", len(df), "with nothing dropped)")
    if cols is None:
        print("columns:     ", Xa.shape[1], "(a numpy array has no names; keep them with pd.DataFrame(..., columns=LAB_COLUMNS))")
    else:
        print("columns:     ", Xa.shape[1], cols, "" if cols == LAB_COLUMNS else "(not LAB_COLUMNS: check the order and the names)")
    # Was the model fit on this X? Its inertia, recomputed on X, must match the inertia it stored.
    same = (hasattr(model, "labels_") and len(model.labels_) == Xa.shape[0]
            and getattr(model, "n_features_in_", None) == Xa.shape[1])
    if same:
        gap = Xa - model.cluster_centers_[model.labels_]
        same = bool(np.isclose(float((gap ** 2).sum()), model.inertia_, rtol=1e-4))
    if not same:
        print("WARNING: this model was not fit on this X. Pass the X it was fit on, or fit it on this X, then check again.")
        return
    sd = Xa.std(axis=0)
    mean = Xa.mean(axis=0)
    scaled = bool(np.allclose(mean, 0, atol=1e-6) and np.allclose(sd, 1, atol=1e-3))
    if scaled:
        print("scaled:       yes, every column has mean 0 and sd", round(float(sd.mean()), 4))
    else:
        print("scaled:       no, the columns' sds run from", f"{sd.min():,.2f}", "to", f"{sd.max():,.2f}")
    print("k:           ", p["n_clusters"], "(the graded fit uses", K, ")")
    print("n_init:      ", p["n_init"], "(should be", N_INIT, ")")
    print("random_state:", p["random_state"], "(should be", SEED, ")")
    sizes = pd.Series(cluster_numbers(model)).value_counts().sort_index()
    print("sizes:       ", ", ".join(f"cluster {c}: {n:,}" for c, n in sizes.items()),
          "(1 = the highest median annual_spend)")


print("helpers ready: cluster_numbers(model), check_fit(model, X)")

# Exercise 1: Distance and scaling (30 points)

Two accounts are alike when their numbers are close. k-means measures close with the
**Euclidean distance**: take the difference in each column, square it, add the squares, and take
the square root. Here are two accounts, C-01417 and C-02031, on the six lab columns.

In [ ]:
PAIR = ["C-01417", "C-02031"]
pair = df.set_index("customer_id").loc[PAIR, LAB_COLUMNS]
print(pair.T)

In [ ]:
# TODO (Exercise 1a): paste the code here, from an AI or the Lab 12 page.
# A prompt that works: "pair is a pandas DataFrame with two rows, one per account, and six columns.
# Compute the Euclidean distance between the two rows as pair_raw. Then compute spend_share:
# the squared difference in annual_spend divided by the sum of the six squared differences.
# Print a table with each column's difference, squared difference and share of the total."
pair_raw = None      # the distance between the two accounts, in raw units
spend_share = None   # the share of the squared distance that comes from annual_spend, 0 to 1

In [ ]:
print("raw distance:", None if pair_raw is None else round(float(pair_raw), 1))
print("share from annual_spend:", None if spend_share is None else f"{float(spend_share):.1%}")

**Scale the columns.** A **z-score** says how many standard deviations a value sits from its
column's mean: (value minus mean) divided by sd. After scaling, every column has mean 0 and sd 1,
so a step of 1 means the same thing in every column. `StandardScaler` does this with the
population sd. Keep the result as a DataFrame with the column names, so `check_fit` can read them.

In [ ]:
# TODO (Exercise 1b): paste the code here, from an AI or the Lab 12 page.
# A prompt that works: "Set X_raw = df[LAB_COLUMNS]. Scale it with sklearn's StandardScaler and keep
# the result as a DataFrame named X_scaled, with columns=LAB_COLUMNS and index=df.index. Then compute
# pair_scaled, the Euclidean distance between the rows of X_scaled for the two customer_ids in PAIR,
# and print each column's share of the squared scaled distance."
X_raw = None        # df[LAB_COLUMNS], unscaled
X_scaled = None     # the same six columns as z-scores
pair_scaled = None  # the distance between the two accounts after scaling

In [ ]:
# Validate: every scaled column should have mean 0 and sd 1.
if X_scaled is not None:
    Xs = pd.DataFrame(np.asarray(X_scaled, dtype=float), columns=LAB_COLUMNS)
    print(pd.DataFrame({"mean": Xs.mean().round(4), "sd": Xs.std(ddof=0).round(4)}))
    print()
    print("scaled distance:", None if pair_scaled is None else round(float(pair_scaled), 3))
else:
    print("Make X_scaled above first.")

**Fit k-means twice at k = 4:** once on the raw columns and once on the scaled ones. Same k,
same `n_init`, same seed. Only the scaling differs.

In [ ]:
# TODO (Exercise 1c): paste the code here, from an AI or the Lab 12 page.
# A prompt that works: "Fit sklearn KMeans(n_clusters=K, n_init=N_INIT, random_state=SEED) on X_raw
# and name it km_unscaled. Fit the same settings on X_scaled and name it km_scaled."
km_unscaled = None   # k-means on the raw columns
km_scaled = None     # k-means on the scaled columns

In [ ]:
# Validate both fits.
for name, model, X in (("km_unscaled", km_unscaled, X_raw), ("km_scaled", km_scaled, X_scaled)):
    print("=====", name)
    if model is None or X is None:
        print("not fit yet")
    else:
        check_fit(model, X)
    print()

In [ ]:
# Unscaled against scaled: what sets the clusters apart? Median of each column by cluster.
for name, model in (("km_unscaled", km_unscaled), ("km_scaled", km_scaled)):
    if model is not None and hasattr(model, "labels_"):
        t = df[LAB_COLUMNS].groupby(cluster_numbers(model)).median().round(2)
        t.insert(0, "n", pd.Series(cluster_numbers(model)).value_counts().sort_index().to_numpy())
        t.index.name = "cluster"
        print("=====", name)
        print(t.to_string())
        print()

**Interpret.**

**Write:** Three to five sentences.

- Give the share of the pair's squared raw distance that comes from `annual_spend`, and say why one column can carry that much.
- Say what changed when you scaled: the scaled distance, and which columns now carry most of it.
- Compare the unscaled and scaled cluster sizes, and say what separates the unscaled clusters.

*Graded on (14 points):* The spend share and the scaled distance from your output. A reason tied to the columns' units. The columns that carry the scaled distance. The unscaled sizes, and what the unscaled clusters have in common.

**Why unscaled is a sort by spend (Exercise 1)**

_Replace this line with your answer._

# Exercise 2: Choose k and read the clusters (40 points)

k-means does not choose k. You do, and two numbers help.

- **Inertia** is the sum of each account's squared distance to its own cluster's center. Smaller
  means tighter clusters. It keeps falling as k rises, because each extra center can only shorten distances.
- The **silhouette** compares each account's average distance to the other accounts in its own cluster
  with its average distance to the accounts in the next nearest cluster, averaged over every account. It runs from -1 to 1. Near 1, the clusters
  are tight and well apart. Near 0, they overlap.

Neither is a rule. Each is evidence for an argument.

In [ ]:
# TODO (Exercise 2a): paste the code here, from an AI or the Lab 12 page.
# A prompt that works: "For k from 2 to 8, fit KMeans(n_clusters=k, n_init=N_INIT, random_state=SEED)
# on X_scaled. Record k, the fit's inertia_, and silhouette_score(X_scaled, labels). Put the results in
# a DataFrame named elbow with columns k, inertia and silhouette, one row per k, and print it."
elbow = None   # one row per k from 2 to 8: k, inertia, silhouette

In [ ]:
# The elbow and silhouette charts, from your elbow table.
if elbow is not None:
    try:
        print(elbow.to_string(index=False))
        import matplotlib.pyplot as plt
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
        ax1.plot(elbow["k"], elbow["inertia"], marker="o", color="#890000")
        ax1.set_xlabel("k")
        ax1.set_ylabel("inertia (lower is tighter)")
        ax1.set_title("Elbow")
        ax2.plot(elbow["k"], elbow["silhouette"], marker="o", color="#4F4F4F")
        ax2.set_xlabel("k")
        ax2.set_ylabel("silhouette (-1 to 1)")
        ax2.set_title("Silhouette")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("(no chart:", type(e).__name__, ")")
else:
    print("Make the elbow table above first.")

**Argue for k.**

**Write:** Three or four sentences.

- What inertia does as k rises, and why the lowest inertia cannot choose k.
- Which k has the highest silhouette, and how strong that is on a scale from -1 to 1, where values near 0 mean the clusters overlap.
- The k you would pick for the credit team, and why. The graded fit stays at k = 4 either way.

*Graded on (8 points):* Why falling inertia cannot pick k by its lowest value. The silhouette read from your table, with the highest k and a fair word on its strength. A k with a reason, which may be a business reason. Any k earns full credit if the argument uses the table.

**Your argument for k (Exercise 2)**

_Replace this line with your answer._

**The graded fit is `km_scaled`, k = 4,** whatever k you argued for. Fixing k is what lets
everyone's clusters compare. The cell below adds each account's cluster number to `df`.

In [ ]:
if km_scaled is not None and hasattr(km_scaled, "labels_"):
    df["cluster"] = cluster_numbers(km_scaled)
    print(df["cluster"].value_counts().sort_index().rename("accounts").to_string())
else:
    print("Fit km_scaled in Exercise 1 first.")

In [ ]:
# TODO (Exercise 2b): paste the code here, from an AI or the Lab 12 page.
# A prompt that works: "Group df by the cluster column and take the median of each column in
# LAB_COLUMNS. Name the result profile, with one row per cluster (the index is the cluster number)
# and one column per lab column. Add a column n with the number of accounts in each cluster, and
# print profile rounded to 2 places."
profile = None   # one row per cluster: n and the median of each lab column

In [ ]:
# The segment mix of each cluster, for context. segment was not a clustering column.
if "cluster" in df.columns:
    print(pd.crosstab(df["cluster"], df["segment"]))

**Name the clusters.**

**Write:** One line per cluster, then one sentence.

- **Clusters 1 to 4:** the size, the medians that set the cluster apart from the other three, and a business name a credit manager would recognize
- **The numbers:** one sentence on what the cluster numbers do and do not tell you

*Graded on (8 points):* Four lines, each with the size and the medians that set the cluster apart, from your profile table. Business names that match those medians. A correct sentence on what the numbering means.

**Name your four clusters (Exercise 2)**

- **Cluster 1:** _size, the medians that set it apart, your name for it_
- **Cluster 2:** _size, the medians that set it apart, your name for it_
- **Cluster 3:** _size, the medians that set it apart, your name for it_
- **Cluster 4:** _size, the medians that set it apart, your name for it_
- **The numbers:** _one sentence_

## The check after the fact

The clusters are built. Now bring `defaulted` back and ask whether the groups k-means found differ
in how often they defaulted. k-means never saw this column, so the answer is a check on what the
clusters found, not a prediction.

In [ ]:
df["defaulted"] = held_back["defaulted"]
base_rate = df["defaulted"].mean()
print(f"{int(df['defaulted'].sum()):,} of {len(df):,} accounts defaulted, a base rate of {base_rate:.1%}.")

In [ ]:
# TODO (Exercise 2c): paste the code here, from an AI or the Lab 12 page.
# A prompt that works: "Group df by cluster and take the mean of defaulted, as a Series named
# default_by_cluster indexed by cluster number. Print it as percentages."
default_by_cluster = None   # the share of each cluster's accounts that defaulted

In [ ]:
if default_by_cluster is not None:
    try:
        t = pd.DataFrame({"accounts": df["cluster"].value_counts().sort_index(),
                          "default rate": pd.Series(default_by_cluster).astype(float)})
        t["vs base rate"] = t["default rate"] - base_rate
        print(t.to_string(formatters={"default rate": "{:.1%}".format, "vs base rate": "{:+.1%}".format}))
    except Exception as e:
        print("default_by_cluster should be a Series indexed by cluster number. This one gave", type(e).__name__)
else:
    print("Make default_by_cluster above first.")

**Interpret.**

**Write:** Four to six sentences.

- The default rate by cluster against the 19.9% base rate, with the numbers. Which cluster stands out, and which of its medians might explain it.
- The deck's eight-column clusters ran from 17.8% to 22.9%. Yours differ. Which two columns did this lab drop, and why does dropping them change what k-means finds?
- Why this is a check on the clusters and not a default model.

*Graded on (12 points):* Default rates by cluster from your output, compared with 19.9%. The cluster that stands out, tied to its medians. The two dropped columns named, with a reason tied to what they measure and how k-means weighs every column it is given. Why rates on clusters built without defaulted are a check, not a prediction.

**The check after the fact (Exercise 2)**

_Replace this line with your answer._

# Exercise 3: Accounts that fit nowhere (30 points)

Week 6 flagged values far from one column's middle. Now do it across all six columns at once.
An account's **distance to its own cluster's center**, in scaled units, says how far it sits from
the accounts it was grouped with. The accounts farthest from their centers fit no group well.
That makes them **worth a look**. It does not make them wrong.

In [ ]:
# TODO (Exercise 3a): paste the code here, from an AI or the Lab 12 page.
# A prompt that works: "km_scaled.transform(X_scaled) gives each account's distance to every cluster
# center. For each account, take the distance to the center of its own cluster (km_scaled.labels_)
# and keep the result as a numpy array named dist_to_center, one value per row of df."
dist_to_center = None   # each account's distance to its own cluster's center, scaled units

In [ ]:
if dist_to_center is not None:
    d = pd.Series(np.asarray(dist_to_center, dtype=float))
    print(d.describe().round(3).to_string())
    try:
        import matplotlib.pyplot as plt
        cut = d.quantile(0.99)
        fig, ax = plt.subplots(figsize=(7, 3.4))
        ax.hist(d, bins=60, color="#BDBDBD")
        ax.axvline(cut, ls="--", color="#890000", label="the top 1% start here")
        ax.set_xlabel("distance to own cluster center (scaled units)")
        ax.set_ylabel("accounts")
        ax.legend(frameon=False)
        plt.show()
    except Exception as e:
        print("(no chart:", type(e).__name__, ")")
else:
    print("Make dist_to_center above first.")

**The top 1%.** Flag the 22 accounts farthest from their own centers (1% of
2,200). For each, find its **most extreme column**: the lab column where the account's z-score, its
row of `X_scaled`, is largest in absolute value. A z-score of 6 on `annual_spend` means the
account's spend sits six standard deviations above the average account's. The most extreme column
is the account's most unusual column overall. It is not always the column that puts the account
far from its own center.

In [ ]:
# TODO (Exercise 3b): paste the code here, from an AI or the Lab 12 page.
# A prompt that works: "Make a DataFrame named flags with the 22 accounts that have the largest
# dist_to_center (1% of the rows of df), sorted from the largest distance down. Columns:
# customer_id, segment, cluster, distance (the dist_to_center value), extreme_col (the column of
# X_scaled with the largest absolute value in that account's row), extreme_z (that z-score, with its
# sign), extreme_value (that column's raw value in df), and defaulted. Print it."
flags = None   # the top 1% by distance, one row per account, with its most extreme column

In [ ]:
if flags is not None:
    print(flags.to_string(index=False))
    print()
    if "extreme_col" in getattr(flags, "columns", []):
        print("the flagged accounts' most extreme columns:")
        print(flags["extreme_col"].value_counts().to_string())
    else:
        print("flags has no extreme_col column yet. Ask the AI to add it, with the name extreme_col.")
else:
    print("Make flags above first.")

In [ ]:
# TODO (Exercise 3c): write this one line yourself.
flag_default_rate = None   # the share of the flagged accounts that defaulted, 0 to 1

In [ ]:
if flag_default_rate is not None:
    print(f"flagged accounts: {float(flag_default_rate):.1%} defaulted, against a base rate of {base_rate:.1%}")
else:
    print("Set flag_default_rate above first.")

**Decide what to review.**

**Write:** About 150 to 250 words, as a short list or a paragraph.

- Three accounts from your flag list, each by `customer_id`, with its most extreme column and what that value is.
- For each, what you would ask and of whom, and why that person or record can answer it.
- One flagged account that is probably unusual for a harmless reason, and why.
- The flagged accounts' default rate against the 19.9% base rate, and what it says about unusual and risky.

*Graded on (18 points):* Three flagged accounts by id, each with its most extreme column and value from your output. A specific question for each, put to a person or record that can confirm the answer on its own. One harmless explanation that fits the numbers. The flagged default rate from your output against 19.9%, read with care.

**Three accounts to review (Exercise 3)**

_Replace this line with your answer._

# Final check

Run the cell below. You do not edit it. It prints `RESULTS`, the numbers grading reads. Use the
**Table of Contents** (the list icon at the top left in Colab) to jump back to an exercise.

In [ ]:
# ===== Final check. Run it. You do not edit this cell. =====
import json
from IPython.display import display, HTML


def final_check():
    """Collect RESULTS from your work and show what is left."""
    g = globals()

    def num(x, places=4):
        try:
            return None if x is None else round(float(x), places)
        except Exception:
            return None

    def fitted(m, width):
        return (m is not None and hasattr(m, "labels_") and hasattr(m, "cluster_centers_")
                and len(m.labels_) == len(df) and getattr(m, "n_features_in_", None) == width)

    def settings_ok(m):
        p = m.get_params()
        return p.get("n_clusters") == 4 and p.get("n_init") == 10 and p.get("random_state") == 12

    def sizes(m):
        try:
            return [int(v) for v in pd.Series(cluster_numbers(m)).value_counts().sort_index().tolist()]
        except Exception:
            return None

    def by_cluster(obj, places):
        try:
            s = pd.Series(obj)
            return {str(int(k)): num(v, places) for k, v in s.items()}
        except Exception:
            return None

    Xs = g.get("X_scaled")
    km_u, km_s = g.get("km_unscaled"), g.get("km_scaled")
    elbow_df, profile_df = g.get("elbow"), g.get("profile")
    dist, flags_df = g.get("dist_to_center"), g.get("flags")

    elbow_out = None
    try:
        elbow_out = [[int(r["k"]), num(r["inertia"], 1), num(r["silhouette"], 3)]
                     for _, r in elbow_df.sort_values("k").iterrows()]
    except Exception:
        elbow_out = None

    medians = None
    try:
        medians = {str(int(c)): {col: num(profile_df.loc[c, col], 2) for col in LAB_COLUMNS}
                   for c in profile_df.index}
    except Exception:
        medians = None

    flag_ids = drivers = None
    try:
        flag_ids = sorted(str(x) for x in flags_df["customer_id"])
        drivers = {str(k): int(v) for k, v in sorted(flags_df["extreme_col"].value_counts().items())}
    except Exception:
        flag_ids = drivers = None

    results = {
        "columns": list(LAB_COLUMNS),
        "pair_raw": num(g.get("pair_raw"), 1),
        "pair_spend_share": num(g.get("spend_share")),
        "pair_scaled": num(g.get("pair_scaled"), 3),
        "sizes_unscaled": sizes(km_u) if fitted(km_u, len(LAB_COLUMNS)) else None,
        "sizes": sizes(km_s) if fitted(km_s, len(LAB_COLUMNS)) else None,
        "inertia_k4": num(km_s.inertia_, 1) if fitted(km_s, len(LAB_COLUMNS)) else None,
        "elbow": elbow_out,
        "medians": medians,
        "base_rate": num(df["defaulted"].mean()) if "defaulted" in df.columns else None,
        "default_rate": by_cluster(g.get("default_by_cluster"), 4) if g.get("default_by_cluster") is not None else None,
        "n_flagged": None if flags_df is None else int(len(flags_df)),
        "flagged_ids": flag_ids,
        "drivers": drivers,
        "flag_default_rate": num(g.get("flag_default_rate")),
    }

    def ok_scaled():
        a = np.asarray(Xs, dtype=float)
        return (a.shape == (len(df), len(LAB_COLUMNS)) and np.allclose(a.mean(axis=0), 0, atol=1e-6)
                and np.allclose(a.std(axis=0), 1, atol=1e-3))

    def ok_pair():
        return None not in (results["pair_raw"], results["pair_spend_share"], results["pair_scaled"]) \
            and 0 <= results["pair_spend_share"] <= 1

    def ok_fits():
        return (fitted(km_u, len(LAB_COLUMNS)) and fitted(km_s, len(LAB_COLUMNS)) and settings_ok(km_u)
                and settings_ok(km_s) and np.abs(km_u.cluster_centers_).max() > 100
                and np.abs(km_s.cluster_centers_).max() < 100)

    def ok_elbow():
        return elbow_out is not None and [e[0] for e in elbow_out] == list(range(2, 9)) \
            and all(e[1] is not None and e[2] is not None for e in elbow_out)

    def ok_profile():
        return medians is not None and sorted(medians) == [str(c) for c in range(1, 4 + 1)] \
            and all(v is not None for m in medians.values() for v in m.values())

    def ok_rates():
        d = results["default_rate"]
        return d is not None and sorted(d) == [str(c) for c in range(1, 4 + 1)] \
            and all(v is not None and 0 <= v <= 1 for v in d.values())

    def ok_dist():
        a = np.asarray(dist, dtype=float).ravel()
        return a.shape == (len(df),) and bool((a >= 0).all())

    def ok_flags():
        need = {"customer_id", "cluster", "distance", "extreme_col", "extreme_z", "defaulted"}
        return (flags_df is not None and need <= set(flags_df.columns)
                and len(flags_df) == round(0.01 * len(df))
                and set(flags_df["extreme_col"]) <= set(LAB_COLUMNS))

    def ok_flag_rate():
        v = results["flag_default_rate"]
        return v is not None and 0 <= v <= 1

    checks = [
        ("X_scaled: 2,200 rows, the six lab columns, mean 0 and sd 1",     "Exercise 1", ok_scaled),
        ("pair_raw, spend_share and pair_scaled: the two accounts",          "Exercise 1", ok_pair),
        ("km_unscaled and km_scaled: k = 4, n_init = 10, random_state = 12", "Exercise 1", ok_fits),
        ("elbow: k from 2 to 8, inertia and silhouette",                     "Exercise 2", ok_elbow),
        ("profile: four clusters, a median for each lab column",             "Exercise 2", ok_profile),
        ("default_by_cluster: a default rate for each cluster",              "Exercise 2", ok_rates),
        ("dist_to_center: one distance per account",                         "Exercise 3", ok_dist),
        ("flags: the top 1%, with the most extreme column of each",          "Exercise 3", ok_flags),
        ("flag_default_rate: the flagged accounts' default rate",            "Exercise 3", ok_flag_rate),
    ]
    rows, passed, todo = [], 0, []
    for label, where, test in checks:
        try:
            good = bool(test())
        except Exception:
            good = False
        if good:
            passed += 1
            rows.append("<tr><td style='font-size:18px;color:#176b3a'>&#10003;</td>"
                        "<td style='padding:2px 14px'>{}</td>"
                        "<td style='color:#176b3a'>done</td></tr>".format(label))
        else:
            todo.append("{} ({})".format(label, where))
            rows.append("<tr><td style='font-size:18px;color:#BE0000'>&#10007;</td>"
                        "<td style='padding:2px 14px'>{}</td>"
                        "<td style='color:#BE0000'>not yet, see <b>{}</b></td></tr>".format(label, where))
    color = "#176b3a" if passed == len(checks) else "#BE0000"
    display(HTML(
        "<div style=\"font-family:Arial,Helvetica,sans-serif\">"
        "<h3 style='margin:4px 0;color:{}'>Lab 12 check: {} of {} done</h3>"
        "<table style='border-collapse:collapse;font-size:14px'>{}</table>"
        "<p style='font-size:12.5px;color:#555;margin-top:8px'>A green row means that piece ran "
        "and has the right shape. It does not grade your numbers or your writing. Grading reruns "
        "the same columns, scaling, k and seed from the data, checks RESULTS against them, and "
        "reads your five written cells.</p></div>".format(color, passed, len(checks), "".join(rows))))
    print("RESULTS =", json.dumps(results))
    if todo:
        print("Still to do:", "; ".join(todo))
    return results


RESULTS = final_check()

## Before you submit

1. **Runtime > Restart session and run all**, so every cell shows its output and the Final check reads your latest work.
2. The Final check prints `RESULTS` with no `null` in it, and every row is green.
3. Your five written cells have your words in them, not the placeholder line.
4. **File > Download > Download .ipynb.** Rename it `<uID>_lab_12.ipynb`, with your uID in place of `<uID>`, and upload it to Lab 12 in Canvas.